In [ ]:
from stable_baselines3 import DQN
import json
import cat_toy_env
import gym_agents


with open("../cat-game/public/common.json") as f:
    env_config = json.load(f)



model = DQN("MlpPolicy", env, verbose=1)


In [ ]:
model = DQN.load("models/sb/cat")
env_kwargs=dict(
  render_mode="",
  config=env_config,
  max_steps = 1000,
  chaser= gym_agents.cat.Cat,
  runners = [gym_agents.toy.Toy, gym_agents.dummy.Dummy],
  reset_interval=2000
)
env = cat_toy_env.CatToyEnv(**env_kwargs)
model.set_env(env)  # 必要に応じて新しい環境をセット
model.learn(total_timesteps=50000, log_interval=4)
model.save("models/sb/cat")  # 上書き保存

In [ ]:

env_kwargs=dict(
  render_mode="human",
  config=env_config,
  max_steps = 1000,
  chaser= gym_agents.cat.Cat,
  runners = [gym_agents.toy.Toy, gym_agents.dummy.Dummy],
  reset_interval=2000
)
env = cat_toy_env.CatToyEnv(**env_kwargs)
model = DQN.load("models/sb/cat")

obs, info = env.reset()
while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()

In [ ]:
import torch
import dqn_onnx
import importlib
importlib.reload(dqn_onnx)

# 7次元のダミー入力（相対位置2, chaser速度2, runner速度2, fatigue1）
obs = torch.randn(1, 7).detach().cpu()  # shape: (1, 7)
#concat_input = obs.repeat(config["cat"]["dqn"]["rnn"]["sequence_length"], 1).unsqueeze(0)  # shape: (1, sequence_length, 7)

# モデルのロード
policy_net = dqn_onnx.DQNOnnx(model.policy)

# ONNX エクスポート
torch.onnx.export(
    policy_net,
    (obs),  # RNN用の入力は (入力テンソル, 隠れ状態) とする
    "cat_dqn_policy.onnx",
    export_params=True,
    opset_version=17,
    input_names=["obs"],
    output_names=["option", "action"],#, "info"],
    dynamic_axes={
        "obs": {0: "batch_size"},  # 観測データのバッチ次元を可変に
        "option": {0: "batch_size"},
        "action": {0: "batch_size"},
    },
    training=torch.onnx.TrainingMode.TRAINING  # ここを変更
)

In [ ]:
model.policy(obs, deterministic=True)